# 04 — Gold set & evaluation harness
**Project:** Clinical Medication Extraction | **Phase 3 of the roadmap**

## Why this is the most important notebook in the project

Everything before this was *building*. This is *measuring* — and measurement is what separates a portfolio project from a demo.

Consider what you can say after each notebook:
- After 03: *"I built a medication extractor."* — so has everyone.
- After 04: *"My rules baseline gets F1 0.83 at drug level, 0.71 with status. Its dominant failure is negation scope, at 3.5% of output. Here's the annotated gold set and the harness that produced those numbers."*

The second sentence is what a team-lead panel is listening for, because it demonstrates the thing they're actually hiring: **the ability to know whether something works.** Anyone can produce output. Far fewer can tell you how good it is, on what evidence, with what uncertainty.

**This is also where your background is a genuine advantage.** Annotation design, sampling, agreement statistics, and measurement error are your professional territory. Most ML engineers treat the gold set as a chore to get past; you know it's an instrument, and that instruments need validation.

**By the end:** a sampling frame, written annotation guidelines, an annotation template, a tested evaluation harness, and a plan for measuring your own consistency.

## Setup

In [2]:
import pandas as pd
import numpy as np
from collections import Counter

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    pass

BASE = '/content/drive/MyDrive/Clinical_notes/' if IN_COLAB else ''
WORK, SRC, GOLD = BASE + 'working/', BASE + 'src/', BASE + 'gold/'

import os, sys
os.makedirs(GOLD, exist_ok=True)
sys.path.insert(0, SRC)

work = pd.read_parquet(WORK + 'notes_subset.parquet')
ex = pd.read_parquet(WORK + 'extractions_rules.parquet')
print(f'{len(work)} notes | {len(ex)} rule extractions')

Mounted at /content/drive
373 notes | 1540 rule extractions


---
# Part 1 — Sampling design

## The trap, stated plainly

The tempting move is to annotate notes where the extractor found something — they're richer, annotation feels productive.

**Doing that makes recall unmeasurable.** Every drug the extractor missed *entirely* lives in the notes where it found nothing. Sample only from its hits and you have built a gold set that is structurally incapable of revealing false negatives. Your recall would not be optimistic; it would be meaningless.

This is selection on an outcome-related variable — the same error as choosing a trial cohort using a variable downstream of the outcome. You'd catch it instantly in a study protocol. Catch it here too.

**The design:** stratify by extraction count so all three strata are represented, then sample proportionally within each.

In [3]:
per_note = ex.groupby('note_id').size().rename('n_ex')
frame = work.join(per_note).fillna({'n_ex': 0})
frame['stratum'] = pd.cut(frame['n_ex'], bins=[-1, 0, 4, 10**6],
                          labels=['none', 'low', 'high'])

print(frame['stratum'].value_counts().reindex(['none','low','high']).to_string())
print()
print('Proportions in the corpus:')
print((frame['stratum'].value_counts(normalize=True)
       .reindex(['none','low','high']) * 100).round(1).to_string())

stratum
none     91
low     145
high    137

Proportions in the corpus:
stratum
none    24.4
low     38.9
high    36.7


In [4]:
N_GOLD = 75
alloc = (frame['stratum'].value_counts(normalize=True).reindex(['none','low','high'])
         * N_GOLD).round().astype(int)
alloc['high'] += N_GOLD - alloc.sum()          # fix rounding drift
print('Allocation:'); print(alloc.to_string())

rng = np.random.RandomState(42)
picks = []
for stratum, n in alloc.items():
    pool = frame[frame['stratum'] == stratum]
    picks.append(pool.sample(n=min(n, len(pool)), random_state=rng))
gold_sample = pd.concat(picks).sort_index()

print()
print(f'Gold sample: {len(gold_sample)} notes')
print(gold_sample['stratum'].value_counts().reindex(['none','low','high']).to_string())

Allocation:
stratum
none    18
low     29
high    28

Gold sample: 75 notes
stratum
none    18
low     29
high    28


### Two choices worth defending in an interview

**Proportional, not equal, allocation.** Equal allocation (25/25/25) would give tighter estimates *per stratum*, but the headline F1 would no longer describe the corpus — you'd have over-weighted the empty notes. With one number to report, representativeness wins. If you later need a reliable estimate on the `none` stratum specifically, that's a separate targeted sample, not a redesign of this one.

**Fixed seed (42).** The sample is reproducible: anyone running this notebook gets the same 75 notes, and your reported metrics are verifiable. Unreproducible evaluation sets are how "we got 0.91" quietly becomes unfalsifiable.

**What 75 buys you.** With ~400 gold events, an F1 around 0.8 has a 95% interval of roughly ±0.04. Precise enough to distinguish 0.83 from 0.91 in Phase 5; not precise enough to argue about 0.83 vs 0.85. Knowing which comparisons your sample can actually support is the difference between reporting results and over-reading them.

---
# Part 2 — Annotation guidelines

**Write these before annotating, not during.** Guidelines written mid-stream get adjusted to fit whatever case is in front of you, and your gold set ends up encoding drift rather than a standard. You know this from protocol work — the pre-registration exists for a reason.

Every ambiguous case you resolve on the fly and don't write down is a decision your future self will make differently in note 60 than in note 6.

### Annotation guidelines v1 — medication events

**Unit of annotation:** one row per *distinct medication event* per note. Two mentions of the same drug in the same role = one row. The same drug in two different roles (active + allergy) = two rows.

**What to annotate**
- Prescription drugs, OTC drugs, insulin, and named supplements (`vitamin D`, `calcium`)
- Drugs in allergy lists — with `status = allergy`
- Drugs mentioned as stopped, held, or refused — with the appropriate status

**What NOT to annotate**
- Drug *classes* without a named agent (`a beta blocker`, `pain medication`)
- IV fluids (`normal saline`, `D5W`) and blood products
- Anesthetic agents used intra-operatively
- Drugs attributed to another person (family history) → skip entirely for v1

**Fields**

| Field | Rule |
|---|---|
| `normalized` | generic name, lowercase. Map brands yourself (`Lasix` → `furosemide`). Combination products hyphenated (`oxycodone-acetaminophen`) |
| `status` | one of: `active`, `discharge`, `allergy`, `historical`, `planned`, `negated`, `inpatient`, `mentioned` |
| `dose` | as written, with unit (`80 mg`). Blank if absent. Do NOT infer |
| `frequency` | canonical form (`daily`, `twice daily`, `at bedtime`, `as needed`). Blank if absent |

**Status decision rules (the hard part — resolve once, here)**
- Currently taking, per the medications section → `active`
- Prescribed at discharge → `discharge`
- Listed as an allergy or intolerance → `allergy`
- Taken in the past, explicitly stopped → `historical`
- Recommended or to be started → `planned`
- Given during this admission only → `inpatient`
- Explicitly negated (`not taking`, `denies`) → `negated`
- Mentioned with no clear role → `mentioned`

**Tie-breakers**
- Drug appears in both `MEDICATIONS` and the plan → annotate both rows
- Dose ranges (`5–10 mg`) → record as written
- Unsure between two statuses → pick the more conservative (`mentioned`) and flag the row in `notes`

**Version:** v1, frozen before annotation begins. Changes require a new version and re-review of affected rows.

---
# Part 3 — Annotation workflow

## Pre-annotation: convenient, and it biases you

You *could* pre-fill the template with the extractor's output and just correct it. It's much faster.

**It also biases recall upward.** Reviewing a list, your attention goes to what's on it. Drugs the extractor missed are exactly the ones you're least likely to notice — the same anchoring effect that makes reviewing someone else's data extraction feel easier and catch less.

**Recommended:** annotate from scratch, note by note. 75 notes at ~2 minutes each is about 2.5 hours — one evening, or five short sessions. That's the price of a recall estimate you can defend.

The cell below exports two files: the notes to read, and an empty template to fill in Google Sheets.

In [5]:
# 1. Notes to read, one per row, in annotation order
gold_notes = gold_sample[['sample_name', 'medical_specialty', 'transcription']].copy()
gold_notes.index.name = 'note_id'
gold_notes.to_csv(GOLD + 'gold_notes_to_annotate.csv')

# 2. Empty annotation template
template = pd.DataFrame(columns=['note_id','normalized','status','dose','frequency','notes'])
template.to_csv(GOLD + 'gold_template.csv', index=False)

print('Wrote:')
print(' ', GOLD + 'gold_notes_to_annotate.csv', f'({len(gold_notes)} notes)')
print(' ', GOLD + 'gold_template.csv')
print()
print('Workflow: open gold_template.csv in Google Sheets, keep the notes file open')
print('alongside, and add one row per medication event as you read.')

Wrote:
  /content/drive/MyDrive/Clinical_notes/gold/gold_notes_to_annotate.csv (75 notes)
  /content/drive/MyDrive/Clinical_notes/gold/gold_template.csv

Workflow: open gold_template.csv in Google Sheets, keep the notes file open
alongside, and add one row per medication event as you read.


In [6]:
# Reading pane — run repeatedly, changing i, to annotate note by note
def show(i):
    row = gold_notes.iloc[i]
    print(f'--- [{i+1}/{len(gold_notes)}]  note_id={row.name}  {row["sample_name"].strip()} ---\n')
    print(row['transcription'])

show(0)

--- [1/75]  note_id=1280  Speech Therapy - Discharge Summary ---

LONG-TERM GOALS:,  Both functional and cognitive-linguistic ability to improve safety and independence at home and in the community.  This goal has been met based on the patient and husband reports the patient is able to complete all activities, which she desires to do at home.  During the last reevaluation, the patient had a significant progress and all cognitive domains evaluated, which are attention, memory, executive functions, language, and visuospatial skill.  She continues to have an overall mild cognitive-linguistic deficit, but this is significantly improved from her initial evaluation, which showed severe impairment., ,The patient does no longer need a skilled speech therapy because she has accomplished all of her goals and her progress has plateaued.  The patient and her husband both agreed with the patient's discharge.


## Measuring your own consistency

Standard practice is two annotators and Cohen's kappa. You have one annotator, so the equivalent is **intra-annotator agreement**: re-annotate 15 of the 75 notes at least a week after the first pass, blind to your original answers, and compare.

**Why this matters and isn't busywork:** it puts a ceiling on your metrics. If you disagree with yourself on 12% of events, then no model can meaningfully score above ~0.88 against this gold set — the remaining gap is measurement noise, not model error. Reporting a model at F1 0.94 against a gold set with 12% self-disagreement is reporting a number your instrument cannot support.

Almost nobody does this in an ML portfolio. You should, precisely because you'll be the rare candidate who can explain *why* it changes how the headline number should be read.

In [7]:
SELF_CHECK_N = 15
recheck = gold_sample.sample(SELF_CHECK_N, random_state=7).index.tolist()
pd.Series(recheck, name='note_id').to_csv(GOLD + 'self_check_notes.csv', index=False)
print(f'{SELF_CHECK_N} notes flagged for blind re-annotation (>=1 week later):')
print(recheck[:8], '...')

15 notes flagged for blind re-annotation (>=1 week later):
[1383, 3277, 3410, 1343, 3340, 1339, 1298, 3284] ...


---
# Part 4 — The evaluation harness

## Design decisions, each with a reason

**1. Matching key = (note_id, normalized drug name).** Not character offsets. Span-level matching is stricter and standard in NER benchmarks, but our task is *event extraction* — what matters is whether we found that the patient is on atorvastatin, not whether we highlighted the exact same characters. Choosing the matching granularity that reflects the task, and saying so, is a real evaluation-design decision.

**2. Greedy one-to-one matching.** A prediction consumes a gold event; neither can be matched twice. Without this, three predictions of the same drug would count as three true positives against one gold event — inflating precision for a duplicate-generating system.

**3. Two strictness levels.** `drug` (did we find the right drug?) and `drug+status` (did we also get its clinical role right?). Reporting both is not hedging — the gap between them *is* the finding. A large gap says extraction works and contextualization doesn't, which points at a specific component.

**4. Attribute accuracy scored only on matched pairs.** Dose accuracy on drugs you failed to find is undefined; conflating the two hides which part is broken. And we skip pairs where both gold and prediction are empty — otherwise "correctly found nothing" inflates accuracy on an attribute that's usually absent.

In [8]:
def _key(r, level):
    base = (r['note_id'], str(r['normalized']).lower().strip())
    return base if level == 'drug' else base + (str(r['status']).lower().strip(),)


def match(pred_df, gold_df, level='drug'):
    """Greedy 1-1 matching. Returns (tp_pairs, fp_indices, fn_indices)."""
    pool = {}
    for i, r in gold_df.iterrows():
        pool.setdefault(_key(r, level), []).append(i)
    tp, fp, used = [], [], set()
    for j, r in pred_df.iterrows():
        candidates = pool.get(_key(r, level), [])
        hit = next((g for g in candidates if g not in used), None)
        if hit is None:
            fp.append(j)
        else:
            used.add(hit)
            tp.append((j, hit))
    fn = [i for i in gold_df.index if i not in used]
    return tp, fp, fn


def prf(tp, fp, fn):
    p = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) else 0.0
    r = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return {'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f, 3),
            'tp': len(tp), 'fp': len(fp), 'fn': len(fn)}


def evaluate(pred_df, gold_df):
    levels = {}
    for level in ['drug', 'drug+status']:
        levels[level] = prf(*match(pred_df, gold_df, level))

    tp, fp, fn = match(pred_df, gold_df, 'drug')
    attrs = {}
    for a in ['dose', 'frequency', 'status']:
        num = den = 0
        for j, i in tp:
            g, p = gold_df.loc[i, a], pred_df.loc[j, a]
            gv = None if pd.isna(g) or str(g).strip() == '' else str(g).lower().strip()
            pv = None if pd.isna(p) or str(p).strip() == '' else str(p).lower().strip()
            if gv is None and pv is None:
                continue                      # both empty -> not informative
            den += 1
            num += int(gv == pv)
        attrs[a] = {'accuracy': round(num / den, 3) if den else None, 'n': den}
    return {'levels': levels, 'attributes': attrs, 'matches': (tp, fp, fn)}

## Test the harness before trusting it

**A metric function is code, and untested code is wrong code.** The failure mode here is nasty: a buggy scorer produces plausible numbers and nothing looks broken. So we score a tiny case whose answer we can verify by hand.

Five predictions, five gold events, with deliberate errors planted: one spurious drug, one missed drug, and two status errors.

In [9]:
gold_toy = pd.DataFrame([
    {'note_id':1,'normalized':'atorvastatin','status':'active','dose':'80 mg','frequency':'daily'},
    {'note_id':1,'normalized':'pantoprazole','status':'active','dose':'25 mg','frequency':'daily'},
    {'note_id':1,'normalized':'sulfamethoxazole-trimethoprim','status':'allergy','dose':None,'frequency':None},
    {'note_id':2,'normalized':'warfarin','status':'active','dose':None,'frequency':None},
    {'note_id':2,'normalized':'metoprolol','status':'active','dose':'50 mg','frequency':'twice daily'},
])
pred_toy = pd.DataFrame([
    {'note_id':1,'normalized':'atorvastatin','status':'active','dose':'80 mg','frequency':'daily'},
    {'note_id':1,'normalized':'pantoprazole','status':'planned','dose':'25 mg','frequency':'daily'},   # status wrong
    {'note_id':1,'normalized':'sulfamethoxazole-trimethoprim','status':'allergy','dose':None,'frequency':None},
    {'note_id':1,'normalized':'vitamin','status':'active','dose':None,'frequency':None},               # spurious
    {'note_id':2,'normalized':'warfarin','status':'negated','dose':None,'frequency':None},             # status wrong
    # metoprolol missing entirely                                                                       # false negative
])

res = evaluate(pred_toy, gold_toy)
print('drug level :', res['levels']['drug'])
print('drug+status:', res['levels']['drug+status'])
print('attributes :', res['attributes'])

drug level : {'precision': 0.8, 'recall': 0.8, 'f1': 0.8, 'tp': 4, 'fp': 1, 'fn': 1}
drug+status: {'precision': 0.4, 'recall': 0.4, 'f1': 0.4, 'tp': 2, 'fp': 3, 'fn': 3}
attributes : {'dose': {'accuracy': 1.0, 'n': 2}, 'frequency': {'accuracy': 1.0, 'n': 2}, 'status': {'accuracy': 0.5, 'n': 4}}


### Verify by hand — this is the check that matters

**Drug level.** 5 predictions, 5 gold events. Matched: atorvastatin, pantoprazole, sulfa, warfarin = **4 TP**. Unmatched prediction: `vitamin` = **1 FP**. Unmatched gold: metoprolol = **1 FN**.
Precision = 4/5 = 0.8. Recall = 4/5 = 0.8. F1 = 0.8. ✓ matches the output.

**Drug+status.** Only atorvastatin (active) and sulfa (allergy) agree on both fields → **2 TP**, 3 FP, 3 FN → 0.4. ✓

**Status accuracy** = 2 correct of 4 matched pairs = 0.5. ✓

The harness is correct. Now — and only now — is it safe to point at real data.

**Note the gap in the toy result: 0.8 at drug level, 0.4 with status.** That's the pattern to watch for in your real numbers. It would mean the extractor finds drugs well and assigns their clinical role badly — which points squarely at the section mapping and negation logic, not the lexicon. **A metric that tells you which component to fix is worth ten metrics that tell you how good you are.**

## Error analysis helper

The metric is a summary; the error table is where the work happens. This produces your Phase 4 comparison categories.

In [10]:
def error_report(pred_df, gold_df, top_n=10):
    res = evaluate(pred_df, gold_df)
    tp, fp, fn = res['matches']

    print('=== FALSE POSITIVES (predicted, not in gold) ===')
    if fp:
        print(pred_df.loc[fp, ['note_id','normalized','status']].head(top_n).to_string(index=False))
        print()
        print('by drug:', Counter(pred_df.loc[fp, 'normalized']).most_common(5))
    print()
    print('=== FALSE NEGATIVES (in gold, missed) ===')
    if fn:
        print(gold_df.loc[fn, ['note_id','normalized','status']].head(top_n).to_string(index=False))
        print()
        print('by drug:', Counter(gold_df.loc[fn, 'normalized']).most_common(5))
    print()
    print('=== STATUS CONFUSIONS (drug matched, status differs) ===')
    conf = Counter((gold_df.loc[i,'status'], pred_df.loc[j,'status'])
                   for j, i in tp if gold_df.loc[i,'status'] != pred_df.loc[j,'status'])
    for (g, p), n in conf.most_common(top_n):
        print(f'  gold={g:12} predicted={p:12} n={n}')
    return res

_ = error_report(pred_toy, gold_toy)

=== FALSE POSITIVES (predicted, not in gold) ===
 note_id normalized status
       1    vitamin active

by drug: [('vitamin', 1)]

=== FALSE NEGATIVES (in gold, missed) ===
 note_id normalized status
       2 metoprolol active

by drug: [('metoprolol', 1)]

=== STATUS CONFUSIONS (drug matched, status differs) ===
  gold=active       predicted=planned      n=1
  gold=active       predicted=negated      n=1


---
# Part 5 — Scoring your real extractor

Run this **after** annotating. The cell handles the not-yet-annotated case so the notebook runs end to end today.

In [13]:
GOLD_FILE = GOLD + 'gold_v1.csv'

if os.path.exists(GOLD_FILE):
    gold = pd.read_csv(GOLD_FILE)

    # --- validate the annotation file before scoring it ---
    required = {'note_id','normalized','status'}
    missing = required - set(gold.columns)
    assert not missing, f'gold_v1.csv missing columns: {missing}'
    VALID_STATUS = {'active','discharge','allergy','historical','planned',
                    'negated','inpatient','mentioned'}
    bad = set(gold['status'].str.lower().unique()) - VALID_STATUS
    assert not bad, f'invalid status values: {bad}'
    unknown = set(gold['note_id']) - set(gold_sample.index)
    assert not unknown, f'note_ids not in the gold sample: {unknown}'
    print(f'Validated: {len(gold)} gold events across {gold["note_id"].nunique()} notes')

    pred = ex[ex['note_id'].isin(gold_sample.index)].reset_index(drop=True)
    print(f'Predictions on those notes: {len(pred)}')
    print()
    res = error_report(pred, gold)
    print()
    print('BASELINE SCORE (rules)')
    for k, v in res['levels'].items():
        print(f'  {k:12} P={v["precision"]:.3f}  R={v["recall"]:.3f}  F1={v["f1"]:.3f}')
    pd.DataFrame(res['levels']).T.to_csv(WORK + 'baseline_scores.csv')
else:
    print(f'{GOLD_FILE} not found — annotate first, then re-run this cell.')
    print('The harness above is tested and ready.')

Validated: 264 gold events across 41 notes
Predictions on those notes: 326

=== FALSE POSITIVES (predicted, not in gold) ===
 note_id                normalized  status
    1311       hydrochlorothiazide  active
    1311                amlodipine  active
    1311                  atenolol  active
    1328             levothyroxine  active
    1328              esomeprazole  active
    1328                 celecoxib  active
    1328                   aspirin  active
    1328                 donepezil  active
    1328 hydrocodone-acetaminophen  active
    1328                penicillin allergy

by drug: [('potassium', 12), ('aspirin', 8), ('insulin', 7), ('calcium', 6), ('lisinopril', 6)]

=== FALSE NEGATIVES (in gold, missed) ===
 note_id   normalized     status
    1298  hydroxyurea     active
    1298    vitamin d     active
    1298 saw palmetto     active
    1298    vitamin c     active
    1304 tetracycline  inpatient
    1306     naproxen    planned
    1330  hydroxyurea historica

In [12]:
harness_src = '''"""Evaluation harness for medication extraction."""
import pandas as pd
from collections import Counter


def _key(r, level):
    base = (r["note_id"], str(r["normalized"]).lower().strip())
    return base if level == "drug" else base + (str(r["status"]).lower().strip(),)


def match(pred_df, gold_df, level="drug"):
    pool = {}
    for i, r in gold_df.iterrows():
        pool.setdefault(_key(r, level), []).append(i)
    tp, fp, used = [], [], set()
    for j, r in pred_df.iterrows():
        candidates = pool.get(_key(r, level), [])
        hit = next((g for g in candidates if g not in used), None)
        if hit is None:
            fp.append(j)
        else:
            used.add(hit)
            tp.append((j, hit))
    fn = [i for i in gold_df.index if i not in used]
    return tp, fp, fn


def prf(tp, fp, fn):
    p = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) else 0.0
    r = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(f, 3),
            "tp": len(tp), "fp": len(fp), "fn": len(fn)}


def evaluate(pred_df, gold_df):
    levels = {lv: prf(*match(pred_df, gold_df, lv)) for lv in ["drug", "drug+status"]}
    tp, fp, fn = match(pred_df, gold_df, "drug")
    attrs = {}
    for a in ["dose", "frequency", "status"]:
        num = den = 0
        for j, i in tp:
            g, p = gold_df.loc[i, a], pred_df.loc[j, a]
            gv = None if pd.isna(g) or str(g).strip() == "" else str(g).lower().strip()
            pv = None if pd.isna(p) or str(p).strip() == "" else str(p).lower().strip()
            if gv is None and pv is None:
                continue
            den += 1
            num += int(gv == pv)
        attrs[a] = {"accuracy": round(num / den, 3) if den else None, "n": den}
    return {"levels": levels, "attributes": attrs, "matches": (tp, fp, fn)}
'''

with open(SRC + 'evaluation.py', 'w') as f:
    f.write(harness_src)

import importlib.util
spec = importlib.util.spec_from_file_location('evaluation', SRC + 'evaluation.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
assert mod.evaluate(pred_toy, gold_toy)['levels']['drug']['f1'] == 0.8
print('Wrote src/evaluation.py and verified it reproduces the toy result.')

Wrote src/evaluation.py and verified it reproduces the toy result.


---
## What you built, and what to do next

**Built:** a stratified sampling frame that keeps recall measurable, frozen annotation guidelines, an annotation workflow, a hand-verified evaluation harness with two strictness levels and attribute accuracy, an error-report function, and a self-agreement protocol.

**Your actual next task — the only manual work in the whole project:**
1. Open `gold_notes_to_annotate.csv` and `gold_template.csv` side by side.
2. Annotate 75 notes from scratch. ~2.5 hours; five sessions of 30 minutes beats one marathon (annotation quality degrades with fatigue, and you'll be measuring exactly that in the self-check).
3. Save as `gold_v1.csv`, re-run Part 5, get your baseline number.
4. Diary any case the guidelines didn't cover — those become v2 and are excellent interview material.

**The four ideas that transfer:**
1. **Never sample your evaluation set using your system's output.** It makes the errors you most need to see invisible.
2. **Freeze guidelines before annotating.** Otherwise you encode drift and call it a standard.
3. **Test the scorer on a case you can verify by hand.** A buggy metric produces plausible numbers, which is the worst kind of wrong.
4. **Report the gap between strictness levels.** It localizes the failure to a component instead of a vibe.

**For `decisions.md`:**
- Gold set n=75, stratified by extraction count (none/low/high), proportional allocation, seed 42 — zero-extraction notes included so recall is measurable
- Matching is event-level on (note, normalized drug), greedy 1-1 — not span offsets; task is event extraction
- Two strictness levels reported (drug, drug+status); the gap localizes contextualization failures
- Attribute accuracy computed only on matched pairs, skipping both-empty — avoids inflation
- Annotation guidelines v1 frozen pre-annotation; 15-note blind re-annotation planned for intra-annotator agreement, which sets the ceiling on reportable model performance
- Single annotator is a documented limitation of this project

**Next: `05_transformer_ner.ipynb`** — a clinical NER model run through this exact harness. Because the harness already exists and is trusted, the comparison is a fair fight, and that's the whole point of doing this notebook first.